In [ ]:
import uproot
import numpy as np
import os
from pathlib import Path

In [ ]:
# Config
DATA_DIR = "./" 
infile = Path(DATA_DIR) / "trkqual_tree_v2.0_training.root"
treename = "trkqualtree"

# Load once
arrays = uproot.open(infile)[treename].arrays(library="np")

def save_perturbed(base_arrays, suffix, transform_fn):
    a = dict(base_arrays)             # copy dict of branch arrays
    transform_fn(a)                   # mutate selected branches
    out = infile.with_name(infile.stem + suffix + infile.suffix)
    with uproot.recreate(out) as f:
        f[treename] = a
    return out, a

# Define all edits in one place
perturbed = [
    ("_nactive_plus1", lambda a: a.__setitem__("trk.nactive", a["trk.nactive"] + 1)),
    ("_momerr_plus10pct", lambda a: a.__setitem__("trk_ent.momerr", a["trk_ent.momerr"] * 1.10)),
    ("_momerr_minus10pct", lambda a: a.__setitem__("trk_ent.momerr", a["trk_ent.momerr"] * 0.90)),
     ("_momerr_plus50pct", lambda a: a.__setitem__("trk_ent.momerr", a["trk_ent.momerr"] * 1.50)),
        ("_momerr_minus50pct", lambda a: a.__setitem__("trk_ent.momerr", a["trk_ent.momerr"] * 0.50)),
    #("_fambig_full", lambda a: a.__setitem__("trk.nnullambig", a["trk.nactive"].copy())),
    #("_momerr_x2", lambda a: a.__setitem__("trk_ent.momerr", a["trk_ent.momerr"] * 2.0)),
    #("_fambigFull_momerrX2", lambda a: (a.__setitem__("trk.nnullambig", a["trk.nactive"].copy()),
                                         #a.__setitem__("trk_ent.momerr", a["trk_ent.momerr"] * 2.0))),   
]

# Quick original checks
print(f"Original nactive mean: {arrays['trk.nactive'].mean():.4f}")
print(f"Original momerr mean:  {arrays['trk_ent.momerr'].mean():.4f}")

# Build all files
written = []
for suffix, fn in perturbed:
    out, arr_out = save_perturbed(arrays, suffix, fn)
    written.append((suffix, out, arr_out))
    print(f"Saved {suffix}: {out}")

# verification table
print("\nVerification:")
print(f"{'perturbed':24s} {'nactive_mean':>12s} {'momerr_mean':>12s}")
print("-" * 52)
print(f"{'ORIGINAL':24s} {arrays['trk.nactive'].mean():12.4f} {arrays['trk_ent.momerr'].mean():12.4f}")
for suffix, _, a in written:
    print(f"{suffix:24s} {a['trk.nactive'].mean():12.4f} {a['trk_ent.momerr'].mean():12.4f}")

Original nactive mean: 32.2548
Original momerr mean:  0.1635
Saved _nactive_plus1: /Users/malikfarouh/Documents/ML workspace/data/trkqual_tree_v2.0_training_nactive_plus1.root
Saved _momerr_plus10pct: /Users/malikfarouh/Documents/ML workspace/data/trkqual_tree_v2.0_training_momerr_plus10pct.root
Saved _momerr_minus10pct: /Users/malikfarouh/Documents/ML workspace/data/trkqual_tree_v2.0_training_momerr_minus10pct.root
Saved _momerr_plus50pct: /Users/malikfarouh/Documents/ML workspace/data/trkqual_tree_v2.0_training_momerr_plus50pct.root
Saved _momerr_minus50pct: /Users/malikfarouh/Documents/ML workspace/data/trkqual_tree_v2.0_training_momerr_minus50pct.root

Verification:
perturbed                nactive_mean  momerr_mean
----------------------------------------------------
ORIGINAL                      32.2548       0.1635
_nactive_plus1                33.2548       0.1635
_momerr_plus10pct             32.2548       0.1798
_momerr_minus10pct            32.2548       0.1471
_momerr_plus5